## AVIATION CLEANING

Business Understanding
1. Problem Statement

The aviation industry has experienced numerous accidents over the past several decades, ranging from minor incidents to complete aircraft destruction and loss of life.

Airlines and insurers are particularly interested in understanding which aircraft makes and models are associated with higher or lower risks of catastrophic failure and passenger injury.

This project uses aviation accident data (1983–2023, filtered for modern professional aircraft) to identify patterns in aircraft safety, focusing on fatal/serious injuries and aircraft destruction outcomes.

The goal is to support data-driven decision-making for aircraft selection and insurance risk assessment.

2. Metric of Success

The success of this project will be determined by:

Identifying aircraft makes and models with consistently low injury rates
Identifying aircraft with low likelihood of being destroyed in accidents
Producing statistically reliable comparisons based on sufficient sample sizes (≥ 50 incidents per make)
Understanding key factors that influence accident severity (e.g. weather, engine type, flight phase)

3. Specific Objective
To identify aircraft makes and models associated with low fatal and serious injury rates
To determine which aircraft types are most resistant to complete destruction in accidents
To support insurers and airlines in selecting lower-risk aircraft for operations and investment

4. Other Objectives
To determine whether certain aircraft makes perform better than others in safety outcomes
To analyze how weather conditions influence accident severity
To assess the impact of flight phase (takeoff, landing, cruise, etc.) on accident outcomes
To compare safety differences between small aircraft and larger passenger aircraft
To explore whether engine type or number of engines affects survival outcomes

5. Research Questions

a. Which aircraft makes and models have the lowest injury rates and destruction rates?

b. What aircraft characteristics (make, model, engine type, number of engines) are associated with safer outcomes?

c. How does weather condition affect the severity of aviation accidents?

d. Which phase of flight is most associated with fatal or serious injuries?

e. Are larger passenger aircraft safer than smaller aircraft in terms of survival outcomes?

Data Understanding

The dataset contains historical aviation accident records from 1948 to 2023, detailing aircraft incidents, passenger outcomes, and operational conditions. For this analysis, the dataset was filtered to include only modern, professionally built aircraft (1983 onwards) to ensure relevance to current aviation safety standards.

Each row represents a single aviation accident event and includes information about:

Aircraft characteristics (make, model, engine type, number of engines)
Operational conditions (weather, flight phase, purpose of flight)
Accident outcomes (fatal, serious, minor injuries, aircraft damage)
Passenger survival information
Key Variables in the Dataset
Aircraft Information
Make – Manufacturer of the aircraft (e.g., Boeing, Cessna)
Model – Specific aircraft model
Aircraft.Category – Type of aircraft (filtered to Airplane only)
Engine.Type – Type of propulsion system
Number.of.Engines – Number of engines installed
Accident Outcome Variables
Total.Fatal.Injuries – Number of deaths
Total.Serious.Injuries – Serious injuries sustained
Total.Minor.Injuries – Minor injuries sustained
Total.Uninjured – Surviving passengers

From these, we derive:

total_passengers – Total people involved in the accident
injury_rate – Proportion of serious + fatal injuries
is_destroyed – Whether the aircraft was completely destroyed
Operational Conditions
Weather.Condition – Weather at time of accident (e.g., VMC, IMC)
Broad.phase.of.flight – Flight stage (takeoff, cruise, landing, etc.)
Purpose.of.flight – Reason for flight (commercial, private, etc.)


## Library Imports

In [156]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', '{:.2f}'.format)

## Data Loading and Inspection

In [157]:
# load the data set
df_raw = pd.read_csv('AviationData.csv', encoding='latin1', low_memory=False)
print(f'Raw shape: {df_raw.shape}')
# check data size
df_raw.head(3)

Raw shape: (88889, 31)


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,Injury.Severity,Aircraft.damage,Aircraft.Category,Registration.Number,Make,Model,Amateur.Built,Number.of.Engines,Engine.Type,FAR.Description,Schedule,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,Fatal(2),Destroyed,NaN,NC6404,Stinson,108-3,No,1.00,Reciprocating,NaN,NaN,Personal,NaN,2.00,0.00,0.00,0.00,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,Fatal(4),Destroyed,NaN,N5069P,Piper,PA24-180,No,1.00,Reciprocating,NaN,NaN,Personal,NaN,4.00,0.00,0.00,0.00,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,Fatal(3),Destroyed,NaN,N5142R,Cessna,172M,No,1.00,Reciprocating,NaN,NaN,Personal,NaN,3.00,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007


In [158]:
# Data types
df_raw.dtypes

Event.Id                   object
Investigation.Type         object
Accident.Number            object
Event.Date                 object
Location                   object
Country                    object
Latitude                   object
Longitude                  object
Airport.Code               object
Airport.Name               object
Injury.Severity            object
Aircraft.damage            object
Aircraft.Category          object
Registration.Number        object
Make                       object
Model                      object
Amateur.Built              object
Number.of.Engines         float64
Engine.Type                object
FAR.Description            object
Schedule                   object
Purpose.of.flight          object
Air.carrier                object
Total.Fatal.Injuries      float64
Total.Serious.Injuries    float64
Total.Minor.Injuries      float64
Total.Uninjured           float64
Weather.Condition          object
Broad.phase.of.flight      object
Report.Status 

In [159]:
# Null value counts and percentages
null_summary = pd.DataFrame({'null_count': df_raw.isnull().sum(), 'null_pct': (df_raw.isnull().sum() / len(df_raw) * 100).round(1)})
null_summary.sort_values('null_pct', ascending=False)

,null_count,null_pct
Schedule,76307,85.80
Air.carrier,72241,81.30
FAR.Description,56866,64.00
Aircraft.Category,56602,63.70
Latitude,54507,61.30
Longitude,54516,61.30
Airport.Code,38757,43.60
Airport.Name,36185,40.70
Broad.phase.of.flight,27165,30.60
Publication.Date,13771,15.50


In [160]:
# Summary statistics
df_raw.describe(include='all')

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,Injury.Severity,Aircraft.damage,Aircraft.Category,Registration.Number,Make,Model,Amateur.Built,Number.of.Engines,Engine.Type,FAR.Description,Schedule,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
count,88889,88889,88889,88889,88837,88663,34382,34373,50132,52704,87889,85695,32287,87507,88826,88797,88787,82805.00,81793,32023,12582,82697,16648,77488.00,76379.00,76956.00,82977.00,84397,61724,82505,75118
unique,87951,2,88863,14782,27758,219,25589,27154,10374,24870,109,4,15,79104,8237,12318,2,NaN,12,31,3,26,13590,NaN,NaN,NaN,NaN,4,12,17074,2924
top,20001212X19172,Accident,CEN22LA149,1984-06-30,"ANCHORAGE, AK",United States,332739N,0112457W,NONE,Private,Non-Fatal,Substantial,Airplane,NONE,Cessna,152,No,NaN,Reciprocating,091,NSCH,Personal,Pilot,NaN,NaN,NaN,NaN,VMC,Landing,Probable Cause,25-09-2020
freq,3,85015,2,25,434,82248,19,24,1488,240,67357,64148,27617,344,22227,2367,80312,NaN,69530,18221,4474,49448,258,NaN,NaN,NaN,NaN,77303,15428,61754,17019
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.15,NaN,NaN,NaN,NaN,NaN,0.65,0.28,0.36,5.33,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.45,NaN,NaN,NaN,NaN,NaN,5.49,1.54,2.24,27.91,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,NaN,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.00,NaN,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.00,NaN,NaN,NaN,NaN,NaN,0.00,0.00,0.00,1.00,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.00,NaN,NaN,NaN,NaN,NaN,0.00,0.00,0.00,2.00,NaN,NaN,NaN,NaN


## Data Cleaning

### Filtering: Aircraft Category, Amateur Build, Event Date

We work on a copy so the raw dataframe is preserved.

In [161]:
df = df_raw.copy()

#### Aircraft.Category

The client is only interested in airplanes, we retain only rows explicitly labelled 'Airplane'

In [162]:
print(df['Aircraft.Category'].value_counts(dropna=False))
df = df[df['Aircraft.Category'] == 'Airplane']
print(f'After Aircraft.Category filter: {df.shape}')

Aircraft.Category
NaN                  56602
Airplane             27617
Helicopter            3440
Glider                 508
Balloon                231
Gyrocraft              173
Weight-Shift           161
Powered Parachute       91
Ultralight              30
Unknown                 14
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64
After Aircraft.Category filter: (27617, 31)


#### Amateur.Built

The client only wants professional builds, so we keep rows where Amateur.Built == No. The 102 NaN rows are dropped

In [163]:
print(df['Amateur.Built'].value_counts(dropna=False))
df = df[df['Amateur.Built'] == 'No']
print(f'After Amateur.Built filter: {df.shape}')

Amateur.Built
No     24417
Yes     3183
NaN       17
Name: count, dtype: int64
After Amateur.Built filter: (24417, 31)


#### Event.Date — filter to 1983 onward

The client assumes a maximum 40-year model lifetime.

In [164]:
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')
print(f'Date parse failures: {df["Event.Date"].isnull().sum()}')
cutoff = pd.Timestamp('1983-01-01')
df = df[df['Event.Date'] >= cutoff]
print(f'After date filter (>= 1983): {df.shape}')

Date parse failures: 0
After date filter (>= 1983): (21447, 31)


### Cleaning and Constructing Key Measurables

#### Injury Columns

Four columns capture passenger outcomes: Total.Fatal.Injuries, Total.Serious.Injuries, Total.Minor.Injuries, Total.Uninjured.

Cleaning assumption: NaN values in these columns are imputed with 0. A NaN most likely means no injuries were recorded in that category, which is consistent with the right-skewed distributions (most accidents have zero fatalities or serious injuries).

Derived metric — injury_rate:  
We estimate total passengers on each flight as the sum of all four injury categories. Then:
total_passengers = fatal + serious + minor + uninjured
injury_rate = (fatal + serious) / total_passengers

injury_rate captures the fraction of passengers who suffered a fatal or serious injury. Rows where total_passengers == 0 (no passenger data at all) have injury_rate set to NaN so they don't skew aggregations.

In [165]:
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']

# Impute NaNs with 0
df[injury_cols] = df[injury_cols].fillna(0)

# Derive total passengers and injury rate
df['total_passengers'] = df[injury_cols].sum(axis = 1)
df['injury_rate'] = np.where(df['total_passengers'] > 0, (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['total_passengers'], np.nan)

print(df['injury_rate'].describe())
print(f'NaN injury_rate rows (no passenger data): {df["injury_rate"].isnull().sum()}')

count   20543.00
mean        0.28
std         0.43
min         0.00
25%         0.00
50%         0.00
75%         0.80
max         1.00
Name: injury_rate, dtype: float64
NaN injury_rate rows (no passenger data): 904


#### Aircraft.Damage — Clean and derive is_destroyed

Cleaning tasks:
- Values Unknown are treated as NaN (they provide no useful signal).
- NaN values are left as NaN; they are excluded from destruction rate calculations.

**Derived column — is_destroyed:
Binary flag: 1 if Aircraft.damage == 'Destroyed', else 0. NaN where damage is unknown/missing.

In [166]:
print('Aircraft.damage value counts (before):')  
print(df['Aircraft.damage'].value_counts(dropna=False))

# replace 'Unknown' with NaN
df['Aircraft.damage'] = df['Aircraft.damage'].replace('Unknown', np.nan)

# Binary destruction flag ofthe derived column
df['is_destroyed'] = np.where(df['Aircraft.damage'].isnull(), np.nan, (df['Aircraft.damage'] == 'Destroyed').astype(float))

print('\nis_destroyed value counts:')
print(df['is_destroyed'].value_counts(dropna=False))

Aircraft.damage value counts (before):
Aircraft.damage
Substantial    16990
Destroyed       2316
NaN             1227
Minor            817
Unknown           97
Name: count, dtype: int64

is_destroyed value counts:
is_destroyed
0.00    17807
1.00     2316
NaN      1324
Name: count, dtype: int64


### Investigate the Make Column

Cleaning tasks identified:
1. Case inconsistency — Cessna, CESSNA, cessna are the same so standardise to title-case.
2. Strip all whitespace.
3. Drop NaN Makes
4. Threshold filter — keep only Makes with ≥50 records.

In [167]:
print('Top raw Makes before cleaning:')
print(df['Make'].value_counts(dropna=False).head(30))

Top raw Makes before cleaning:
Make
CESSNA                      4867
PIPER                       2803
Cessna                      2279
Piper                       1186
BOEING                      1037
BEECH                       1018
Beech                        413
MOONEY                       238
Boeing                       227
CIRRUS DESIGN CORP           218
AIR TRACTOR INC              217
AIRBUS                       215
BELLANCA                     158
AERONCA                      149
MAULE                        144
Mooney                       125
EMBRAER                      123
Air Tractor                  117
LUSCOMBE                      95
CHAMPION                      91
DEHAVILLAND                   91
STINSON                       91
AIR TRACTOR                   89
CIRRUS                        80
NORTH AMERICAN                79
GRUMMAN                       77
DIAMOND AIRCRAFT IND INC      74
AVIAT AIRCRAFT INC            72
Maule                         71
Grumman

In [168]:
# 1. Drop NaN Makes
df = df.dropna(subset=['Make'])

# 2. Strip whitespace and convert to title case
df['Make'] = df['Make'].str.strip().str.title()

# Additional known normalizations
make_map = {'Mcdonnell Douglas': 'McDonnell Douglas','De Havilland': 'De Havilland', 'Cirrus Design Corp': 'Cirrus', 'Air Tractor Inc': 'Air Tractor', 'Robinson Helicopter': 'Robinson', 'Robinson Helicopter Co': 'Robinson', 'Socata': 'Socata',}
df['Make'] = df['Make'].replace(make_map)

print('Top Makes after cleaning:')
print(df['Make'].value_counts().head(30))

Top Makes after cleaning:
Make
Cessna                            7146
Piper                             3989
Beech                             1431
Boeing                            1264
Air Tractor                        425
Mooney                             363
Cirrus                             357
Airbus                             243
Bellanca                           219
Maule                              215
Aeronca                            200
Champion                           158
Embraer                            153
Grumman                            147
Luscombe                           141
Stinson                            129
McDonnell Douglas                  108
North American                     106
Dehavilland                         95
Taylorcraft                         93
Aero Commander                      90
Aviat Aircraft Inc                  76
Socata                              75
Diamond Aircraft Ind Inc            74
De Havilland                     

In [169]:
# 3. Threshold filter: keep Makes with >= 50 records
make_counts = df['Make'].value_counts()
valid_makes = make_counts[make_counts >= 50].index
df = df[df['Make'].isin(valid_makes)]
print(f'Unique Makes after threshold filter: {df["Make"].nunique()}')
print(f'Rows remaining: {df.shape[0]}')

Unique Makes after threshold filter: 34
Rows remaining: 17892


### Inspect the Model Column

In [170]:
# Drop NaN models
print(f'NaN models before: {df["Model"].isnull().sum()}')
df = df.dropna(subset=['Model'])
print(f'NaN models after: {df["Model"].isnull().sum()}')

NaN models before: 13
NaN models after: 0


In [171]:
# strip whitespace, upper-case for consistency
df['Model'] = df['Model'].str.strip().str.upper()

# Check whether model labels are unique per make
model_make_counts = df.groupby('Model')['Make'].nunique()
multi_make_models = model_make_counts[model_make_counts > 1]
print(f'Model names shared across multiple Makes: {len(multi_make_models)}')
print(multi_make_models.head(10))

Model names shared across multiple Makes: 93
Model
100      2
112      2
112A     2
140      2
190      2
1900D    2
200      2
320      2
350      2
390      2
Name: Make, dtype: int64


In [172]:
# Model labels are NOT unique per make so:
df['Make_Model'] = df['Make'] + ' — ' + df['Model']
print(f'Unique Make_Model identifiers: {df["Make_Model"].nunique()}')
print(df['Make_Model'].value_counts().head(10))

Unique Make_Model identifiers: 2128
Make_Model
Cessna — 172     769
Boeing — 737     403
Cessna — 152     316
Cessna — 182     304
Cessna — 172S    276
Piper — PA28     273
Cessna — 172N    249
Cirrus — SR22    240
Cessna — 180     213
Cessna — 172M    180
Name: count, dtype: int64


### Cleaning Other Columns

We inspect and clean Engine.Type, Weather.Condition, Number.of.Engines, Purpose.of.flight, and Broad.phase.of.flight.

#### Engine.Type

In [173]:
print(df['Engine.Type'].value_counts(dropna=False))

Engine.Type
Reciprocating      12835
NaN                 3214
Turbo Prop           931
Turbo Fan            701
Unknown              105
Turbo Jet             71
Geared Turbofan       12
Turbo Shaft            9
UNK                    1
Name: count, dtype: int64


In [174]:
# change 'Unknown', 'UNK', 'LR', 'NONE' to NaN  (placeholder or ambiguous values)
df['Engine.Type'] = df['Engine.Type'].replace({'Unknown': np.nan, 'UNK': np.nan, 'LR': np.nan, 'NONE': np.nan})

# Drop categories with very few examples (< 10)
et_counts = df['Engine.Type'].value_counts()
rare_types = et_counts[et_counts < 10].index
df['Engine.Type'] = df['Engine.Type'].replace({t: np.nan for t in rare_types})
print(df['Engine.Type'].value_counts(dropna=False))

Engine.Type
Reciprocating      12835
NaN                 3329
Turbo Prop           931
Turbo Fan            701
Turbo Jet             71
Geared Turbofan       12
Name: count, dtype: int64


#### Weather.Condition

In [175]:
print(df['Weather.Condition'].value_counts(dropna=False))

Weather.Condition
VMC    14295
NaN     2417
IMC      905
Unk      186
UNK       76
Name: count, dtype: int64


In [176]:
# change 'Unk' and 'UNK' to NaN
df['Weather.Condition'] = df['Weather.Condition'].replace({'UNK': np.nan, 'Unk': np.nan})
print(df['Weather.Condition'].value_counts(dropna=False))

Weather.Condition
VMC    14295
NaN     2679
IMC      905
Name: count, dtype: int64


#### Number.of.Engines

In [177]:
print(df['Number.of.Engines'].value_counts(dropna=False))

Number.of.Engines
1.00    13222
2.00     2470
NaN      2089
4.00       67
3.00       26
0.00        5
Name: count, dtype: int64


In [178]:
# 0 engines is not meaningful for an airplane so change to NaN
df['Number.of.Engines'] = df['Number.of.Engines'].replace(0, np.nan)

# Convert to nullable integer
df['Number.of.Engines'] = df['Number.of.Engines'].astype('Int64')
print(df['Number.of.Engines'].value_counts(dropna=False))

Number.of.Engines
1       13222
2        2470
<NA>     2094
4          67
3          26
Name: count, dtype: Int64


#### Purpose.of.flight

In [179]:
print(df['Purpose.of.flight'].value_counts(dropna=False))

Purpose.of.flight
Personal                     9844
NaN                          3047
Instructional                2410
Aerial Application            724
Business                      409
Unknown                       303
Positioning                   269
Skydiving                     157
Aerial Observation            147
Other Work Use                121
Banner Tow                     86
Flight Test                    73
Ferry                          72
Executive/corporate            65
Glider Tow                     29
Public Aircraft - Federal      28
Public Aircraft                27
Public Aircraft - State        21
Air Race show                  15
Firefighting                   12
Public Aircraft - Local        11
PUBS                            3
Air Race/show                   2
Air Drop                        2
ASHO                            2
Name: count, dtype: int64


In [180]:
# change 'Unknown' to NaN
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace('Unknown', np.nan)

# Convert sparse 'Public Aircraft' variants (fewer than 50) into a single category
pub_variants = ['Public Aircraft - Federal', 'Public Aircraft - Local', 'Public Aircraft - State']
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace({v: 'Public Aircraft' for v in pub_variants})
print(df['Purpose.of.flight'].value_counts(dropna=False))

Purpose.of.flight
Personal               9844
NaN                    3350
Instructional          2410
Aerial Application      724
Business                409
Positioning             269
Skydiving               157
Aerial Observation      147
Other Work Use          121
Public Aircraft          87
Banner Tow               86
Flight Test              73
Ferry                    72
Executive/corporate      65
Glider Tow               29
Air Race show            15
Firefighting             12
PUBS                      3
Air Race/show             2
Air Drop                  2
ASHO                      2
Name: count, dtype: int64


#### Broad.phase.of.flight

In [181]:
print(df['Broad.phase.of.flight'].value_counts(dropna=False))

Broad.phase.of.flight
NaN            15427
Landing         1110
Takeoff          425
Cruise           238
Approach         210
Maneuvering      127
Taxi              99
Go-around         81
Descent           62
Climb             52
Standing          35
Unknown           11
Other              2
Name: count, dtype: int64


In [182]:
# change 'Unknown' to NaN 
# change 'Other' to NaN 
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].replace({'Unknown': np.nan, 'Other': np.nan})
print(df['Broad.phase.of.flight'].value_counts(dropna=False))

Broad.phase.of.flight
NaN            15440
Landing         1110
Takeoff          425
Cruise           238
Approach         210
Maneuvering      127
Taxi              99
Go-around         81
Descent           62
Climb             52
Standing          35
Name: count, dtype: int64


### Column Removal

Drop any columns with fewer than 20,000 non-null values.

In [183]:
non_null_counts = df.notnull().sum()
print('Non-null counts per column:')
print(non_null_counts.sort_values())

Non-null counts per column:
Schedule                   2139
Broad.phase.of.flight      2439
Air.carrier                8448
Airport.Code              11648
Airport.Name              11754
Report.Status             14094
Purpose.of.flight         14529
Engine.Type               14550
Weather.Condition         15200
Number.of.Engines         15785
Longitude                 15978
Latitude                  15981
is_destroyed              16733
Aircraft.damage           16733
Publication.Date          17091
injury_rate               17103
Injury.Severity           17162
FAR.Description           17535
Registration.Number       17715
Location                  17875
Country                   17878
Total.Uninjured           17879
total_passengers          17879
Total.Minor.Injuries      17879
Event.Id                  17879
Total.Fatal.Injuries      17879
Amateur.Built             17879
Model                     17879
Make                      17879
Aircraft.Category         17879
Event.Date  

In [184]:
df.columns

Index(['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date',
       'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code',
       'Airport.Name', 'Injury.Severity', 'Aircraft.damage',
       'Aircraft.Category', 'Registration.Number', 'Make', 'Model',
       'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description',
       'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries',
       'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured',
       'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status',
       'Publication.Date', 'total_passengers', 'injury_rate', 'is_destroyed',
       'Make_Model'],
      dtype='object')

In [185]:
cols_to_keep = non_null_counts[non_null_counts >= 20000].index.tolist()

# Always keep our derived and key columns even if they fall below threshold
essential = ['injury_rate', 'is_destroyed', 'total_passengers', 'Make_Model', 'Number.of.Engines', 'Make', 'Weather.Condition', 'Broad.phase.of.flight']
final_cols = list(set(cols_to_keep + essential))
df = df[[c for c in df.columns if c in final_cols]]
print(f'Columns retained: {len(df.columns)}')
print(df.columns.tolist())

Columns retained: 8
['Make', 'Number.of.Engines', 'Weather.Condition', 'Broad.phase.of.flight', 'total_passengers', 'injury_rate', 'is_destroyed', 'Make_Model']


### Final Dataset Summary

In [186]:
df.head()

,Make,Number.of.Engines,Weather.Condition,Broad.phase.of.flight,total_passengers,injury_rate,is_destroyed,Make_Model
4150,Boeing,4,VMC,Taxi,588.00,0.00,0.00,Boeing — 747
4171,Piper,1,IMC,Cruise,2.00,1.00,1.00,Piper — PA-28-140
4285,De Havilland,2,VMC,Standing,5.00,0.20,NaN,De Havilland — DHC-6
6760,Boeing,3,VMC,Taxi,100.00,0.00,0.00,Boeing — 727-200
6806,Beech,1,VMC,Climb,1.00,0.00,0.00,Beech — C35


Save Cleaned Data

In [187]:
df.to_csv('aviation_cleaned.csv', index=False)
print('Saved to aviation_cleaned.csv')

Saved to aviation_cleaned.csv
